# Alloy Discovery Engine — 互動實驗

此 notebook 為薄殼，所有業務邏輯在 `alloy_engine` 套件中。

In [ ]:
import sys
sys.path.insert(0, '..')  # 確保能 import alloy_engine

import torch
from alloy_engine.config import DEFAULT_DEVICE, CHECKPOINT_DIR
from alloy_engine.data.synthetic import generate_training_data
from alloy_engine.features.engineering import composition_to_features_np
from alloy_engine.models.surrogate import SurrogateBundle
from alloy_engine.ga.gpu_ga import GPUGeneticAlgorithm
from alloy_engine.scenarios import SCENARIOS
from alloy_engine.visualization import (
    plot_data_distribution, plot_convergence, plot_composition_heatmap
)

device = DEFAULT_DEVICE
print(f'裝置: {device}')

## 1. 訓練資料分佈

In [ ]:
compositions, tc, hc, br, sigma_y = generate_training_data(n_samples=8000)
plot_data_distribution(tc, hc, br, sigma_y)

## 2. 載入或訓練模型

In [ ]:
checkpoint = CHECKPOINT_DIR / 'bundle.pt'
if checkpoint.exists():
    bundle = SurrogateBundle.load(checkpoint, device=device)
    print('已載入 checkpoint')
else:
    X_features = composition_to_features_np(compositions, device=device)
    bundle = SurrogateBundle.from_trained(
        X_features, tc, hc, br, sigma_y, device=device, epochs=300
    )
    bundle.save(checkpoint)

## 3. 單情境 GA 搜尋

In [ ]:
scenario_name = '中溫廢熱_350C'
cfg = SCENARIOS[scenario_name]

ga = GPUGeneticAlgorithm(
    predict_fn=bundle.predict_properties,
    device=device,
    population_size=50_000,  # 調小以加快 notebook 速度
    **cfg,
)
pop, fit, info = ga.run(n_gen=50, verbose=True)

In [ ]:
top_idx = fit.argsort(descending=True)[:5]
results = {scenario_name: {
    'config':    cfg,
    'top_comps': pop[top_idx].cpu().numpy(),
    'history':   dict(ga.history),
}}
plot_convergence(results)
plot_composition_heatmap(results)